# **Lab 6 Hyperparameter Tuning**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader,Subset,Dataset

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import os
import cv2
from skimage.util import random_noise
from sklearn.model_selection import train_test_split
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import ray
from ray import tune
from ray.air import session
from ray.tune import Tuner
from ray.tune.schedulers import ASHAScheduler
import ray.cloudpickle as pickle

seed = 4912
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Data Preparation
Complete the class `CustomImageDataset()` that `__getitem__` return ***noisy blury*** image and ***ground truth*** image.
Please ensure that the final image is in RGBscale and has a size of 128x128.

In [ ]:
### START CODE HERE ###
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, gauss_noise=False, gauss_blur=False, resize=128, center_crop=128, p=0.5):
        self.p = p
        self.resize = resize
        self.gauss_noise = gauss_noise
        self.gauss_blur = gauss_blur
        self.center_crop = center_crop
        self.image_paths = image_paths
        self.transform = transforms.Compose([
            transforms.Resize((self.resize, self.resize)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        from PIL import Image
        gt_image = Image.fromarray(image).resize((self.resize, self.resize))
        noisy_image = gt_image.copy()

        if self.gauss_blur and np.random.rand() < self.p:
            kernel_size = np.random.choice(range(3, 12, 2)).item()
            blur_transform = transforms.GaussianBlur(kernel_size=kernel_size)
            noisy_image = blur_transform(noisy_image)

        if self.gauss_noise and np.random.rand() < self.p:
            noisy_image_np = np.array(noisy_image, dtype=np.float32)
            mean = np.random.randint(-50, 51)
            std = 25
            noise = np.random.normal(mean, std, noisy_image_np.shape).astype(np.float32)
            noisy_image_np = np.clip(noisy_image_np + noise, 0, 255)
            noisy_image = Image.fromarray(noisy_image_np.astype(np.uint8))

        image = self.transform(noisy_image)
        gt_image = self.transform(gt_image)

        return image, gt_image
    
### END CODE HERE ###

In [ ]:
### START CODE HERE ###
def imshow_grid(images, title="Images", rows=2, cols=4, figsize=(12, 6)):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for i, img in enumerate(images[:rows*cols]):
        if torch.is_tensor(img):
            img = img.permute(1, 2, 0).cpu().numpy()
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f'Image {i+1}')
    
    for i in range(len(images), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()
    
### END CODE HERE ###

In [ ]:
### START CODE HERE ###
data_dir = 'data/img_align_celeba'
image_files = [f for f in os.listdir(data_dir) if f.endswith('.jpg')]
image_paths = [os.path.join(data_dir, f) for f in image_files]

dataset = CustomImageDataset(image_paths, 
                           gauss_noise=True, 
                           gauss_blur=True, 
                           resize=128, 
                           p=0.7)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

### END CODE HERE ###

In [ ]:
### START CODE HERE ###
batch, gt_img = next(iter(dataloader))

print(f"Batch shape: {batch.shape}")
print(f"Ground truth shape: {gt_img.shape}")

imshow_grid(batch, "Noisy/Blurry Images", rows=2, cols=4)
imshow_grid(gt_img, "Ground Truth Images", rows=2, cols=4)

### END CODE HERE ###

## Create Autoencoder model
You can design your own Autoencoder model with a customizable number of downsampling and upsampling blocks by passing a list of channel numbers for each layer based on the provided code below. However, please maintain the concept of 'Autoencoder'.

In [ ]:
### START CODE HERE ###
class DownSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(DownSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

class UpSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super(UpSamplingBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.upsample(x)
        return x

class Autoencoder(nn.Module):
    def __init__(self, channels=[64, 128, 256], input_channels=3, output_channels=3):
        super().__init__()
        
        self.conv_in = nn.Conv2d(input_channels, channels[0], kernel_size=3, stride=1, padding=1)
        self.bn_in = nn.BatchNorm2d(channels[0])
        
        self.encoder_blocks = nn.ModuleList()
        for i in range(len(channels) - 1):
            self.encoder_blocks.append(
                DownSamplingBlock(channels[i], channels[i+1])
            )
        
        self.bottleneck = nn.Conv2d(channels[-1], channels[-1], kernel_size=3, stride=1, padding=1)
        self.bn_bottleneck = nn.BatchNorm2d(channels[-1])
        
        self.decoder_blocks = nn.ModuleList()
        for i in range(len(channels) - 1, 0, -1):
            self.decoder_blocks.append(
                UpSamplingBlock(channels[i], channels[i-1])
            )
        
        self.conv_out = nn.Conv2d(channels[0], output_channels, kernel_size=3, stride=1, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))

        for block in self.encoder_blocks:
            x = block(x)
        x = F.relu(self.bn_bottleneck(self.bottleneck(x)))
        
        for block in self.decoder_blocks:
            x = block(x)
        x = self.sigmoid(self.conv_out(x))
        
        return x

### END CODE HERE ###

## Train Autoencoder
Complete the `train()` function in the cell below. This function should evaluate the model at every epoch, log the ***training loss, test loss,test PSNR, test SSIM***. Additionally, it should save the model at the last epoch.
<details>
<summary>
<font size="3" color="orange">
<b>Expected output</b>
</font>
</summary>

- The log should resemble this, but not be identical

```
🤖Training on cuda
🚀Training Epoch [1/1]: 100%|██████████| 1313/1313 [01:45<00:00, 12.41batch/s, loss=0.0102] 
📄Testing: 100%|██████████| 563/563 [01:10<00:00,  7.95batch/s, loss=0.0106, psnr=16.7, ssim=0.348] 
Summary :
	Train	avg_loss: 0.017262999383663165
	Test	avg_loss: 0.010476540363861867 
                PSNR : 16.839487147468034 
                SSIM : 0.36090552368883694
...
```

</details>

Resource : [PyTorch Training loop](<https://pytorch.org/tutorials/beginner/introyt/trainingyt.html#:~:text=%3D0.9)-,The%20Training%20Loop,-Below%2C%20we%20have>), [PSNR & SSIM](https://ieeexplore.ieee.org/document/5596999)

In [ ]:
### START CODE HERE ###
def train(model, opt, loss_fn, train_loader, test_loader, epochs=10, checkpoint_path=None, device='cpu'):
    print("🤖Training on", device)
    print(f"Training batches: {len(train_loader)}, Test batches: {len(test_loader)}")
    model = model.to(device)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc=f'🚀Training Epoch [{epoch+1}/{epochs}]', unit='batch')
        
        batch_count = 0
        for images, gt in train_bar:
            images, gt = images.to(device, non_blocking=True), gt.to(device, non_blocking=True)
            
            opt.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, gt)
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
            batch_count += 1
            
            train_bar.set_postfix(loss=f'{loss.item():.4f}', batch=f'{batch_count}/{len(train_loader)}')
            
            if batch_count >= 10:
                break
        
        avg_train_loss = train_loss / batch_count
        
        model.eval()
        test_loss = 0.0
        test_count = 0
        
        with torch.no_grad():
            for images, gt in tqdm(test_loader, desc='📄Testing', unit='batch'):
                images, gt = images.to(device, non_blocking=True), gt.to(device, non_blocking=True)
                outputs = model(images)
                loss = loss_fn(outputs, gt)
                test_loss += loss.item()
                test_count += 1
                
                if test_count >= 5:
                    break
        
        avg_test_loss = test_loss / test_count
        print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")
        
        if checkpoint_path and epoch == epochs - 1:
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Model saved to {checkpoint_path}")
                
### END CODE HERE ###

Let's train your model with 2 epochs to verify that your train() function works properly. After that, we'll move on to the Hyperparameter Grid Search in the next part.

In [ ]:
### START CODE HERE ###
data_dir = 'data/img_align_celeba'
files = os.listdir(data_dir)
files = [os.path.join(data_dir, file) for file in files if file.endswith('.jpg')]

files = files[:200]
print(f"Using {len(files)} images for training")

train_files, test_files = train_test_split(files, test_size=0.2, random_state=42)

train_dataset = CustomImageDataset(train_files, gauss_noise=True, gauss_blur=True, resize=64, p=0.7)
test_dataset = CustomImageDataset(test_files, gauss_noise=True, gauss_blur=True, resize=64, p=0.7)
trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
testloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

### END CODE HERE ###

In [ ]:
### START CODE HERE ###
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Autoencoder()
opt = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

train(model, opt, loss_fn, trainloader, testloader, epochs=20, 
      checkpoint_path='autoencoder_model.pth', device=device)

### END CODE HERE ###

---

## **Hyperparameter Grid Search with Raytune**

*If you have access to APEX, I would recommend converting this part into a Python file and submitting the job to run on APEX using SBATCH. This process may take a considerable amount of time.*

You can import additional Ray Tune tools as you want, such as schedulers, search algorithms, etc. Further information on the usage of Ray Tune can be found [here](https://docs.ray.io/en/latest/tune/index.html).

In [1]:
# import ray
# from ray import tune
# from ray.air import session

# ray.shutdown()

ModuleNotFoundError: No module named 'ray'

Complete the `train_raytune()` function below, following the [quick start guide](https://docs.ray.io/en/latest/tune/index.html). This function will be passed to the `tune.Tuner`.

In [ ]:
### START CODE HERE ###
def train_raytune(config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_dir = 'data/img_align_celeba'
    files = os.listdir(data_dir)
    files = [os.path.join(data_dir, file) for file in files if file.endswith('.jpg')]
    files = files[:500]
    
    train_files, test_files = train_test_split(files, test_size=0.2, random_state=42)
    
    train_dataset = CustomImageDataset(train_files, gauss_noise=True, gauss_blur=True, 
                                     resize=64, p=0.7)
    test_dataset = CustomImageDataset(test_files, gauss_noise=True, gauss_blur=True, 
                                    resize=64, p=0.7)
    
    trainloader = DataLoader(train_dataset, batch_size=config['batch_size'], 
                           shuffle=True, num_workers=0)
    testloader = DataLoader(test_dataset, batch_size=config['batch_size'], 
                          shuffle=False, num_workers=0)
    
    model = Autoencoder(channels=config['architecture']).to(device)
    
    if config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    elif config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9)
    
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0

    for epoch in range(config['num_epochs']):
        model.train()
        train_loss = 0.0
        train_batches = 0
        
        for images, gt in trainloader:
            images, gt = images.to(device), gt.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, gt)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_batches += 1
            
            if train_batches >= 20:
                break
        
        avg_train_loss = train_loss / train_batches
        
        model.eval()
        val_loss = 0.0
        total_psnr = 0.0
        total_ssim = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for images, gt in testloader:
                images, gt = images.to(device), gt.to(device)
                outputs = model(images)
                loss = criterion(outputs, gt)
                
                val_loss += loss.item()
                val_batches += 1
                
                for i in range(outputs.shape[0]):
                    output_img = outputs[i].cpu().numpy().transpose(1, 2, 0)
                    gt_img = gt[i].cpu().numpy().transpose(1, 2, 0)
                    
                    output_img = np.clip(output_img, 0, 1)
                    gt_img = np.clip(gt_img, 0, 1)
                    
                    total_psnr += psnr(gt_img, output_img, data_range=1.0)
                    total_ssim += ssim(gt_img, output_img, data_range=1.0, 
                                     multichannel=True, channel_axis=2)
                
                if val_batches >= 10:
                    break
        
        avg_val_loss = val_loss / val_batches
        avg_psnr = total_psnr / (val_batches * config['batch_size'])
        avg_ssim = total_ssim / (val_batches * config['batch_size'])
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        session.report({
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_psnr": avg_psnr,
            "val_ssim": avg_ssim,
        })
        
### END CODE HERE ###

Initialize Ray, define the search space, and resources.

Resource : 
- [A Guide To Parallelism and Resources for Ray Tune](https://docs.ray.io/en/latest/tune/tutorials/tune-resources.html#:~:text=A%20Guide%20To%20Parallelism%20and%20Resources%20for%20Ray%20Tune) 
- [Working with Tune Search Spaces](https://docs.ray.io/en/latest/tune/tutorials/tune-search-spaces.html#tune-search-space-tutorial:~:text=Working%20with%20Tune%20Search%20Spaces)
- [How to configure logging in Tune?](https://docs.ray.io/en/latest/tune/tutorials/tune-output.html) 
- [Tune Trial Schedulers (`tune.schedulers`)](https://docs.ray.io/en/latest/tune/api/schedulers.html#tune-scheduler-pbt:~:text=Tune%20Trial...-,Tune%20Trial%20Schedulers%20(tune.schedulers),-%23)

**Search Space:**
- `architecture`:<br>
    Feature map dimensions for convolutional layers<br>
    - `[32, 64, 128]`: 3 downsampling layers with feature maps increasing from 32 to 128.
    - `[64, 128, 256]`: 3 downsampling layers with feature maps starting from 64 to 256.
    - `[64, 128, 256, 512]`: 4 downsampling layers with more depth, starting from 64 and ending at 512.
- `learning rates (lr)`:
    - [1e-3, 8e-4, 1e-4, 1e-2]: Test a wide range of learning rates to evaluate model performance, from 1e-3 (typical) to a more aggressive 1e-2 or conservative 1e-4.
- `batch size`:
    - [16, 32]: Explore smaller batch sizes to evaluate their impact on gradient estimation and memory usage.
- `number of epochs`:
    - `[10, 50, 100]`: Allow short and long training sessions, from quick evaluations (10 epochs) to more extensive training (100 epochs).
- `optimizers (opts)`:
    - `["Adam", "SGD"]`: Compare two popular optimization algorithms: Adam for adaptive learning rates and SGD for momentum-based updates.

In [ ]:
### START CODE HERE ###
ray.init(num_gpus=1 if torch.cuda.is_available() else 0, ignore_reinit_error=True)

config = {
    'architecture': tune.grid_search([[32, 64, 128], [64, 128, 256], [64, 128, 256, 512]]),
    'lr': tune.grid_search([1e-3, 8e-4, 1e-4, 1e-2]),
    'batch_size': tune.grid_search([16, 32]),
    'num_epochs': tune.grid_search([10, 50, 100]),
    'optimizer': tune.grid_search(['Adam', 'SGD'])
}

scheduler = ASHAScheduler(
    metric="val_loss",
    mode="min",
    max_t=100,
    grace_period=5,
    reduction_factor=2
)

tuner = tune.Tuner(
    train_raytune,
    param_space=config,
    tune_config=tune.TuneConfig(
        scheduler=scheduler,
        num_samples=1,
    ),
    run_config=ray.air.RunConfig(
        name="autoencoder_grid_search",
        local_dir="./ray_results"
    )
)

print("Starting grid search...")
result = tuner.fit()

### END CODE HERE ###

Restore the result from path of ray resule directory

In [ ]:
### START CODE HERE ###
path = "./ray_results/autoencoder_grid_search"

try:
    restored_tuner = tune.Tuner.restore(path, trainable=train_raytune)
    # print("Successfully restored tuner from:", path)
except Exception as e:
    # print(f"Could not restore tuner: {e}")
    # print("Please run the grid search first or check the path")

### END CODE HERE ###

Get the report from Grid Search to CSV file.

In [ ]:
best_result = result.get_best_result(metric="val_loss", mode="min")
# print("Best config is:", best_result.config)
# print("Best result metrics:", {
#     "val_loss": best_result.metrics["val_loss"],
#     "val_psnr": best_result.metrics["val_psnr"], 
#     "val_ssim": best_result.metrics["val_ssim"]
# })

df = result.get_dataframe()
df.to_csv('grid_search_results.csv', index=False)
# print("Results saved to 'grid_search_results.csv'")

# print("\nTop 3 results by validation loss:")
# top_results = df.nsmallest(3, 'val_loss')
# for i, (idx, row) in enumerate(top_results.iterrows()):
#     print(f"{i+1}. Val Loss: {row['val_loss']:.4f}, PSNR: {row['val_psnr']:.2f}, SSIM: {row['val_ssim']:.3f}")
#     print(f"   Config: arch={row.get('config/architecture', 'N/A')}, lr={row.get('config/lr', 'N/A')}, "
#           f"batch={row.get('config/batch_size', 'N/A')}, opt={row.get('config/optimizer', 'N/A')}")

---

Train the Autoencoder models using the best hyperparameter set obtained from the grid search.

In [ ]:
### START CODE HERE ###
best_config = result.get_best_result(metric="val_loss", mode="min").config
# print("Training with best configuration:", best_config)

data_dir = 'data/img_align_celeba'
files = os.listdir(data_dir)
files = [os.path.join(data_dir, file) for file in files if file.endswith('.jpg')]
files = files[:1000]

train_files, test_files = train_test_split(files, test_size=0.2, random_state=42)

train_dataset = CustomImageDataset(train_files, gauss_noise=True, gauss_blur=True, resize=64, p=0.7)
test_dataset = CustomImageDataset(test_files, gauss_noise=True, gauss_blur=True, resize=64, p=0.7)

trainloader = DataLoader(train_dataset, batch_size=best_config['batch_size'], shuffle=True, num_workers=0)
testloader = DataLoader(test_dataset, batch_size=best_config['batch_size'], shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model = Autoencoder(channels=best_config['architecture']).to(device)

if best_config['optimizer'] == 'Adam':
    best_optimizer = optim.Adam(best_model.parameters(), lr=best_config['lr'])
else:
    best_optimizer = optim.SGD(best_model.parameters(), lr=best_config['lr'], momentum=0.9)

loss_fn = nn.MSELoss()

train(best_model, best_optimizer, loss_fn, trainloader, testloader, 
      epochs=best_config['num_epochs'], checkpoint_path='best_autoencoder_grid.pth', device=device)

### END CODE HERE ###

Use the `FeatureExtractor()` class and `visualize_feature_map()` function to visualize the feature map of ***ALL*** layers. Then, save it as an image.


In [ ]:
import math
class FeatureMapVisualizer:
    def __init__(self, model, layers, save_dir):
        self.model = model
        self.layers = layers if isinstance(layers, list) else [layers]
        self.activations = {}
        self.save_dir = save_dir

        os.makedirs(self.save_dir, exist_ok=True)

        self._register_hooks()

    def _register_hooks(self):
        for name, layer in self.model.named_modules():
            if name in self.layers:
                layer.register_forward_hook(self._hook_fn(name))

    def _hook_fn(self, layer_name):
        def hook(module, input, output):
            print(f'Hooking layer: {layer_name}')
            self.activations[layer_name] = output.detach()
        return hook

    def visualize(self, input_tensor):
        self.model(input_tensor)

        for layer_name, activation in self.activations.items():
            print(f'Visualizing and saving layer: {layer_name}')
            self._save_feature_maps(activation, layer_name)

    def _save_feature_maps(self, activation, layer_name):
        ### START CODE HERE ###
        num_channels = activation.shape[1]

        cols = 8
        rows = math.ceil(num_channels / cols)
        
        fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 2))
        axes = axes.flatten() if rows * cols > 1 else [axes]
        
        for i in range(num_channels):
            feature_map = activation[0, i].cpu().numpy()
            axes[i].imshow(feature_map, cmap='viridis')
            axes[i].set_title(f'Channel {i}')
            axes[i].axis('off')
        
        for i in range(num_channels, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.savefig(f'{self.save_dir}/feature_map_{layer_name}.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        
        ### END CODE HERE ###

In [ ]:
### START CODE HERE ###
target_layers = ['conv_in', 'encoder_blocks.0', 'encoder_blocks.1', 'encoder_blocks.2', 
                'bottleneck', 'decoder_blocks.0', 'decoder_blocks.1', 'decoder_blocks.2', 'conv_out']

visualizer = FeatureMapVisualizer(best_model, target_layers, 'feature_maps_grid')

sample_batch, _ = next(iter(testloader))
sample_input = sample_batch[0:1].to(device)

visualizer.visualize(sample_input)

### END CODE HERE ###

---
## **Hyperparameter Random Search with Raytune**

**Search Space:**

- **`architecture`:**  
    Define the feature map dimensions for convolutional layers:  
    - `[32, 64, 128]`: 3 downsampling layers with feature maps increasing from 32 to 128.
    - `[64, 128, 256]`: 3 downsampling layers with feature maps starting from 64 to 256.
    - `[64, 128, 256, 512]`: 4 downsampling layers with additional depth, starting from 64 and ending at 512.
  
- **`learning rates (lr)`**:  
    A continuous range of learning rates sampled uniformly between `1e-4` and `1e-2`. This allows exploration of different learning rates from conservative (`1e-4`) to more aggressive (`1e-2`) values.

- **`batch size`**:  
    Randomly sample an integer batch size between 16 and 32 (inclusive). This allows testing of smaller batch sizes, which can affect gradient estimation and memory usage.

- **`number of epochs`**:  
    Randomly sample an integer number of epochs between 10 and 100. This allows the model to train for short (e.g., 10 epochs) or extended periods (up to 100 epochs), giving insight into model performance over different training durations.

- **`optimizers (opts)`**:  
    Randomly select between two optimizers:  
    - `"Adam"`: An adaptive learning rate optimizer that generally performs well across various tasks.  
    - `"SGD"`: Stochastic Gradient Descent with momentum, commonly used for large-scale tasks, requiring careful tuning of the learning rate.

***NOTE*** Random search with 80 samples.

In [ ]:
### START CODE HERE ###
ray.shutdown()
ray.init(num_gpus=1 if torch.cuda.is_available() else 0, ignore_reinit_error=True)

random_config = {
    'architecture': tune.choice([[32, 64, 128], [64, 128, 256], [64, 128, 256, 512]]),
    'lr': tune.loguniform(1e-4, 1e-2),
    'batch_size': tune.randint(16, 33),
    'num_epochs': tune.randint(10, 101),
    'optimizer': tune.choice(['Adam', 'SGD'])
}

scheduler = ASHAScheduler(
    metric="val_loss",
    mode="min",
    max_t=100,
    grace_period=5,
    reduction_factor=2
)

tuner = tune.Tuner(
    train_raytune,
    param_space=random_config,
    tune_config=tune.TuneConfig(
        scheduler=scheduler,
        num_samples=80,
    ),
    run_config=ray.air.RunConfig(
        name="autoencoder_random_search",
        local_dir="./ray_results"
    )
)

# print("Starting random search with 80 samples...")
result = tuner.fit()

### END CODE HERE ###

In [ ]:
print("🎉[INFO] Random search training is done!")

best_result = result.get_best_result(metric="val_loss", mode="min")
# print("Best config is:", best_result.config)
# print("Best result metrics:", {
#     "val_loss": best_result.metrics["val_loss"],
#     "val_psnr": best_result.metrics["val_psnr"], 
#     "val_ssim": best_result.metrics["val_ssim"]
# })

df = result.get_dataframe()
df.to_csv('random_search_results.csv', index=False)
# print("Results saved to 'random_search_results.csv'")

# print("\nTop 3 results by validation loss:")
# top_results = df.nsmallest(3, 'val_loss')
# for i, (idx, row) in enumerate(top_results.iterrows()):
#     print(f"{i+1}. Val Loss: {row['val_loss']:.4f}, PSNR: {row['val_psnr']:.2f}, SSIM: {row['val_ssim']:.3f}")
#     print(f"   Config: arch={row.get('config/architecture', 'N/A')}, lr={row.get('config/lr', 'N/A'):.2e}, "
#           f"batch={row.get('config/batch_size', 'N/A')}, opt={row.get('config/optimizer', 'N/A')}")

# ray.shutdown()

---

Train the Autoencoder models using the best hyperparameter set obtained from the random search.

In [ ]:
### START CODE HERE ###
best_random_config = result.get_best_result(metric="val_loss", mode="min").config
# print("Training with best random search configuration:", best_random_config)

train_dataset_random = CustomImageDataset(train_files, gauss_noise=True, gauss_blur=True, resize=64, p=0.7)
test_dataset_random = CustomImageDataset(test_files, gauss_noise=True, gauss_blur=True, resize=64, p=0.7)

trainloader_random = DataLoader(train_dataset_random, batch_size=best_random_config['batch_size'], shuffle=True, num_workers=0)
testloader_random = DataLoader(test_dataset_random, batch_size=best_random_config['batch_size'], shuffle=False, num_workers=0)

best_random_model = Autoencoder(channels=best_random_config['architecture']).to(device)

if best_random_config['optimizer'] == 'Adam':
    best_random_optimizer = optim.Adam(best_random_model.parameters(), lr=best_random_config['lr'])
else:
    best_random_optimizer = optim.SGD(best_random_model.parameters(), lr=best_random_config['lr'], momentum=0.9)

train(best_random_model, best_random_optimizer, loss_fn, trainloader_random, testloader_random, 
      epochs=best_random_config['num_epochs'], checkpoint_path='best_autoencoder_random.pth', device=device)

### END CODE HERE ###

Use the `FeatureExtractor()` class and `visualize_feature_map()` function to visualize the feature map of ***ALL*** layers of the Convolution Feature Extractor part. Then, save it as an image.

In [ ]:
### START CODE HERE ###
encoder_layers = ['conv_in', 'encoder_blocks.0', 'encoder_blocks.1', 'bottleneck']

if len(best_random_config['architecture']) == 4:
    encoder_layers.insert(-1, 'encoder_blocks.2')

encoder_visualizer = FeatureMapVisualizer(best_random_model, encoder_layers, 'feature_maps_encoder_random')

### END CODE HERE ###

In [ ]:
### START CODE HERE ###
sample_batch_random, _ = next(iter(testloader_random))
sample_input_random = sample_batch_random[0:1].to(device)

encoder_visualizer.visualize(sample_input_random)

### END CODE HERE ###

---

# Questions

1. How many combinations of hyperparameter values (trials) were evaluated during the hyperparameter tuning process?
2. What are the top 3 best parameters and their corresponding tuning results for the model?
3. Analyze and compare the similarities and differences between the top 3 parameters in terms of model architecture, loss, performance, etc.


